# Apo-density control for FuncBind density fusion

The holo run showed that a residual derived from the real 2Fo-Fc map of
**5MGL / 7MU**
improves a frozen FuncBind. That map contains the crystallographic ligand, so
the improvement could have come from copying the ligand out of the density.

This run repeats the experiment on the same map with the **ligand erased**: the
FuncBind ligand occupancy field (atom radii scaled by
1.25) is used as a soft mask and the vacated voxels are
refilled with the crop's bulk-solvent level (-0.084).
Mean density at the ligand atom centres drops from
1.93 to
-0.08, and the fraction of voxels within
2.0 Å of a ligand atom above 1.5σ
drops from 0.166 to
0.001. Everything outside the mask is
bit-identical to the holo crop.

> **How to read this.** The adapter still fits this one target, so a good apo
> score is *not* evidence that density helped — it is evidence that the adapter
> can reach the target without any ligand density, i.e. that the holo result was
> memorization. The transfer arm makes the same point without retraining: it
> replays the holo-trained adapter on this ligand-free map.


In [ ]:
import json
from pathlib import Path

import numpy as np

search_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    root for root in search_roots
    if (root / 'exps' / 'density_fusion').exists()
)
RESULT_DIR = PROJECT_ROOT / 'exps/density_fusion/apo_5mgl_target69_20260730'
with (RESULT_DIR / 'result.json').open() as handle:
    report = json.load(handle)
report['status'], report['experiment']


## Quantitative result

| Method | Occupancy MSE | mIoU | Pred. atoms | Matched | Precision | Recall | F1 |
|---|---:|---:|---:|---:|---:|---:|---:|
| Holo-density adapter (holo run) | 1.834e-05 | 0.5711 | 10 | 10 | 1.000 | 0.909 | 0.952 |
| Original FuncBind | 3.366e-05 | 0.3756 | 11 | 8 | 0.727 | 0.727 | 0.727 |
| Apo-density adapter | 1.556e-05 | 0.5253 | 12 | 11 | 0.917 | 1.000 | 0.957 |
| Holo-trained adapter on this density | 2.547e-05 | 0.4376 | 10 | 10 | 1.000 | 0.909 | 0.952 |

8 paired chains share one initial latent and a
restored sampler RNG state per arm; the table reports the oracle-best chain of
each arm.


![Metric summary](figures/apo_density_metric_summary.png)


## The density input

> **This run uses apo density: the ligand density is masked out.** The map is
> the measured 2Fo-Fc map of the complex with the ligand's own density removed,
> so the encoder sees the pocket as if nothing were bound. The mask is
> FuncBind's ligand occupancy field (atom radii ×1.25) and the
> vacated voxels are refilled with the crop's bulk-solvent level
> (-0.084) rather than zeroed — zero is not solvent in this
> arcsinh z-scored map. No separately measured apo crystal structure is
> involved.

The crop is a 16 Å cube at 0.25 Å resolution aligned to FuncBind's centered
pocket frame.

![Density slices](figures/apo_density_slices.png)


## 3D occupancy comparison

![3D occupancy comparison](figures/apo_density_occupancy_3d.png)


## Paired diffusion behavior

![Paired latent MSE](figures/apo_density_paired_latent_mse.png)


## Adapter fit

Only the 1,118,720-parameter density
adapter was trained; the FuncBind checkpoints, the INR decoder and the VoxBind
density encoder were frozen.

![Adapter training](figures/apo_density_adapter_training.png)


## Limitations

- The apo map is a masked holo map, not a separately measured apo structure: solvent reorganization and side-chain relaxation on ligand release are not modeled, and density from receptor atoms that fall inside the ligand envelope is erased along with it.
- The density adapter is overfit on this same single target.
- Best-of-chain visualization uses target latent MSE as an oracle.
- This is a proof of concept, not a held-out density-fusion evaluation.
